# Fine-tune VLM 2 — relationships between object pairs

**Prepare and review pairs → train → inspect results → final test.** Change `ACTION` and run the cells. The helper handles loading, saving and GPU cleanup. No training or downloads run by default.

This learns **relationship labels for supplied pairs**, not candidate-pair selection. Initial boxes come from semantic-region proposals and need human review; this is a GT-assisted component experiment, not full ConceptGraphs evaluation. Keep the original scene/layout split from notebook 1. Object-naming adapters remain separate.

In [1]:
# 1. Setup. Use the same kernel/packages as finetune_vlm.ipynb.
from pathlib import Path
import sys, importlib, json
import pandas as pd
from IPython.display import display

ROOT = next((p for p in (Path.cwd(), *Path.cwd().parents)
             if (p / "code/midterm_report/lora_baseline.py").is_file()), None)
if ROOT is None:
    raise RuntimeError("Start Jupyter inside the Senior Capstone repository.")
sys.path.insert(0, str(ROOT / "code/midterm_report"))
import lora_baseline as lb
importlib.reload(lb)
print("Python:", sys.executable)

# Optional installation only; restart the kernel afterwards. No weights downloaded.
INSTALL_PACKAGES = False
if INSTALL_PACKAGES:
    import subprocess
    subprocess.check_call([sys.executable, "-m", "pip", "install",
                           "peft>=0.18", "accelerate>=1.0", "bitsandbytes>=0.48", "num2words", "ipywidgets"])
    raise RuntimeError("Installation finished. Set INSTALL_PACKAGES=False, restart the kernel, then Run All.")

Python: c:\ProgramData\anaconda3\envs\cleanjupyter\python.exe


In [2]:
# 2. Parameters. This replaces the load/train/reload/test boolean switches.
ACTION = "test"                       # "prepare", "train", or "test"
MODEL_KEY = "qwen"                        # Change to "smolvlm" for the other model.
RUN_NAME = "relations_3epochs"            # Same name for train and test; retries get safe suffixes.
# Saved runs: artifacts/relationships/<model>/<run>/; dataset fingerprints stay in metadata.
USE_4BIT = True
LORA = {"rank": 8, "alpha": 16, "dropout": 0.05}
TRAINING = {"max_steps": -1, "epochs": 3, "accumulation": 4,
            "learning_rate": 1e-4, "eval_steps": 10}
# max_steps=20 gives a short pilot; -1 uses epochs. Epoch runs evaluate/save each epoch.

DATASET_NAME = "osma_relation_pairs_v1"
DATASET_DIR = ROOT / "dataset/_training" / DATASET_NAME
DATA_ROOT = ROOT / "dataset/osma-bench/data"
DATA_SETTINGS = {"pairs_per_scene": 12, "image_max_side": 512, "seed": 42}
SOURCE_MANIFEST = None                    # Auto-selects only if exactly ONE object manifest exists.
# Otherwise set to the manifest.json produced by notebook 1. Existing splits are reused.
# Changing dataset settings requires a new DATASET_NAME; review labels are never overwritten.
TEST_RUN = None                           # Auto-select latest completed matching run; or set exact run folder.
# Keep labels/data fixed between both models and final testing. No test scores during training.

In [3]:
# 3. Run. prepare is CPU-only. train loads a fresh base + relationship LoRA,
# measures pretrained validation, trains, saves, measures adapted validation, and frees the GPU.
# test loads the saved relationship adapter and evaluates BOTH versions on the same test pairs.
result = lb.run_relation_pipeline(
    ROOT, DATASET_DIR, model_key=MODEL_KEY, action=ACTION, run_name=RUN_NAME,
    training=TRAINING, lora=LORA, use_4bit=USE_4BIT,
    source_manifest=SOURCE_MANIFEST, data_settings=DATA_SETTINGS, test_run=TEST_RUN,
)

Testing saved relationship adapter: d:\School Stuff\shanghai work\Senior Capstone\artifacts\relationships\qwen\relations_3epochs


Loading weights:   0%|          | 0/625 [00:00<?, ?it/s]

In [4]:
# 4. Results / review. In prepare mode, label pairs in train, validation AND test.
# Start with a smaller reviewed subset if needed; only reviewed usable pairs are used.
# Include verified "none" negatives and positive relations. Unknown != none.
# Review A/B as distinct physical objects, not merely same-class semantic regions.
if ACTION == "prepare":
    display(pd.DataFrame(lb.relation_dataset_summary(DATASET_DIR)["counts"]))
    print("Labels:", DATASET_DIR / "labels.jsonl")
    print("a_on_b = A rests on B; a_in_b = A is inside B. b_* reverses direction.")
    print("next_to = immediate side-by-side neighbors; nearby = physically close but not immediate neighbors.")
    print("Prefer in > on > next_to > nearby. Proximity labels are symmetric; unclear depth means unknown.")
    print("none = none of those relationships; unknown = insufficient evidence / invalid boxes.")
    review_form = lb.review_relation_pairs(DATASET_DIR, DATA_ROOT)
    # Save + next writes immediately. You may edit labels.jsonl instead of using the form.
    # When the reviewed subset is ready, set ACTION='train' and rerun parameters/run/results.
else:
    print("Run:", result["run_dir"])
    display(pd.DataFrame({mode: {k: v for k, v in scores.items() if k != "per_relation"}
                          for mode, scores in result["metrics"].items()}).T)
    for mode, scores in result["metrics"].items():
        print(mode, "— per-relation results")
        display(pd.DataFrame.from_dict(scores["per_relation"], orient="index"))
    # macro_relation_f1 averages supported POSITIVE relations; none is reported separately.
    # weighted_exact_match is SECONDARY: nearby examples weigh 0.5; others weigh 1.
    # This is an experimental reporting preference, not confidence or a change to training loss.
    # Missing relation classes have no evidence: do not claim performance for them.
    # These are fixed-pair classification scores, not whole-graph edge recall or mIoU.
    # Change MODEL_KEY and rerun parameters/run/results to train the other model.
    # After choosing settings on validation, set ACTION='test'. No manual reload is needed.

Run: d:\School Stuff\shanghai work\Senior Capstone\artifacts\relationships\qwen\relations_3epochs


,examples,exact_match,weighted_exact_match,nearby_score_weight,macro_relation_f1,relation_precision,relation_recall,relation_f1,invalid_output_rate,positive_relations_evaluated
pretrained,25.0,0.04,0.057143,0.5,0.000000,0.00,0.000000,0.000000,0.0,3.0
adapted,25.0,0.56,0.400000,0.5,0.239316,0.56,0.736842,0.636364,0.0,3.0


pretrained — per-relation results


,support,precision,recall,f1
a_on_b,0,0.0,NaN,0.000000
b_on_a,1,0.0,0.000000,0.000000
a_in_b,0,0.0,NaN,0.000000
b_in_a,0,0.0,NaN,0.000000
next_to,3,0.0,0.000000,0.000000
nearby,15,0.0,0.000000,0.000000
none,6,1.0,0.166667,0.285714


adapted — per-relation results


,support,precision,recall,f1
a_on_b,0,0.000000,NaN,NaN
b_on_a,1,0.000000,0.000000,0.000000
a_in_b,0,0.000000,NaN,NaN
b_in_a,0,0.000000,NaN,NaN
next_to,3,0.000000,0.000000,0.000000
nearby,15,0.583333,0.933333,0.717949
none,6,0.000000,0.000000,0.000000
